# LangGraph 뼈대(LangChain) 구성

In [ ]:
from typing import List, TypedDict

In [8]:
class GraphState(TypedDict):
    """RAG 파이프라인의 공유 상태.

    Attributes:
        question: 사용자의 질문
        chat_history: 대화 기록
        filters: 검색 필터 (기관, 예산 등)
        retrieved_contexts: 검색된 문서 원문들
        final_answer: 최종 요약 답변
        citations_used: 출처 (파일명 등)
        errors: 에러 메시지
        relevance_score: 검색 문서 관련성 점수 (Self-Corrective RAG 확장용)
        retry_count: 재시도 횟수 (Self-Corrective RAG 확장용)
    """

    question: str
    chat_history: List[str]
    filters: dict
    retrieved_contexts: List[str]
    final_answer: str
    citations_used: List[str]
    errors: List[str]
    relevance_score: float
    retry_count: int

In [ ]:
def retrieve(state: GraphState) -> dict:
    """벡터스토어에서 관련 문서를 검색한다.

    TODO: retriever 담당자가 구현한 모듈을 연결할 것.
    """
    # retriever 담당 모듈 연결 지점
    # from retriever import search
    # docs = search(state["question"], state["filters"])

    return {
        "retrieved_contexts": [],
        "citations_used": [],
    }


def generate(state: GraphState) -> dict:
    """검색된 컨텍스트를 기반으로 답변을 생성한다.

    TODO: prompt + parser 담당자가 구현한 체인을 연결할 것.
    """
    if not state.get("retrieved_contexts"):
        return {
            "final_answer": "관련 문서를 찾을 수 없습니다.",
            "errors": [*state.get("errors", []), "컨텍스트 없음"],
        }

    # prompt 담당 모듈 연결 지점
    # from prompts import GENERATE_PROMPT
    # from parser import parse_answer
    # chain = GENERATE_PROMPT | llm | parse_answer
    # answer = chain.invoke({...})

    return {"final_answer": ""}


def grade_documents(state: GraphState) -> dict:
    """검색된 문서의 관련성을 평가한다. (Self-Corrective RAG 확장용)

    TODO: prompt 담당자가 grading prompt를 구현할 것.
    """
    contexts = state.get("retrieved_contexts", [])
    total = len(contexts)
    score = 1.0 if total > 0 else 0.0

    return {
        "retrieved_contexts": contexts,
        "relevance_score": score,
    }


def transform_query(state: GraphState) -> dict:
    """질문을 재작성하여 검색 품질을 높인다. (Self-Corrective RAG 확장용)

    TODO: prompt 담당자가 rewrite prompt를 구현할 것.
    """
    return {
        "question": state["question"],
        "retry_count": state.get("retry_count", 0) + 1,
    }


def decide_to_generate(state: GraphState) -> str:
    """문서 관련성에 따라 다음 노드를 결정한다.

    Returns:
        "generate" - 관련 문서가 충분하거나 재시도 초과
        "transform_query" - 관련 문서가 부족한 경우
    """
    max_retries = 2

    if state.get("retry_count", 0) >= max_retries:
        return "generate"

    if state.get("relevance_score", 0) >= 0.5:
        return "generate"

    return "transform_query"

In [ ]:
from langgraph.graph import END, StateGraph


def build_simple_graph():
    """MVP용 단순 RAG 그래프.

    Flow: retrieve → generate → END
    """
    graph = StateGraph(GraphState)

    graph.add_node("retrieve", retrieve)
    graph.add_node("generate", generate)

    graph.set_entry_point("retrieve")
    graph.add_edge("retrieve", "generate")
    graph.add_edge("generate", END)

    return graph.compile()


def build_self_corrective_graph():
    """Self-Corrective RAG 그래프.

    Flow:
        retrieve → grade_documents → (조건부)
            ├─ 관련성 충분 → generate → END
            └─ 관련성 부족 → transform_query → retrieve (루프)
    """
    graph = StateGraph(GraphState)

    graph.add_node("retrieve", retrieve)
    graph.add_node("grade_documents", grade_documents)
    graph.add_node("generate", generate)
    graph.add_node("transform_query", transform_query)

    graph.set_entry_point("retrieve")
    graph.add_edge("retrieve", "grade_documents")

    graph.add_conditional_edges(
        "grade_documents",
        decide_to_generate,
        {
            "generate": "generate",
            "transform_query": "transform_query",
        },
    )

    graph.add_edge("transform_query", "retrieve")
    graph.add_edge("generate", END)

    return graph.compile()

In [ ]:
# MVP 실행
result = build_simple_graph().invoke({
    "question": "이 사업의 총 예산은 얼마인가요?",
    "chat_history": [],
    "filters": {},
    "retrieved_contexts": [],
    "final_answer": "",
    "citations_used": [],
    "errors": [],
    "relevance_score": 0.0,
    "retry_count": 0,
})

print(f"질문: {result['question']}")
print(f"답변: {result['final_answer']}")
print(f"출처: {result['citations_used']}")
if result.get("errors"):
    print(f"에러: {result['errors']}")